## Step 10 - OCR với Qwen2.5-VL (drop-in thay EasyOCR / step 6)

Notebook này thay thế hoàn toàn EasyOCR trong `kaggle/06_extract-ocr.ipynb` (và
`data_processing/04_extract_ocr.py` ở local) bằng **Qwen2.5-VL-Instruct** - VLM
có OCR tiếng Việt tốt hơn EasyOCR rất nhiều trên frame broadcast (chữ trên banner,
logo kênh, dải chữ chạy, hàng chữ nhỏ trên nền phức tạp).

Khác notebook 09 (làm OCR + caption cùng lúc):
- **Chỉ OCR** - prompt gọn hơn → model tập trung, JSON parse ổn định hơn
- **Output cùng format step 6** → có thể dùng thay thế trực tiếp, không cần đụng backend

**Chạy song song trên 2 GPU T4**: model load 1 lần trong MỖI tiến trình worker con
(`cuda:0` và `cuda:1`), video chia round-robin cho 2 worker → throughput ~2x.
`device_map="auto"` không đáng dùng: model 3B (~7GB) nằm gọn 1 T4 nên không trải ra
được, chỉ thêm phí PCIe mà không nhanh hơn.

**Input / output y hệt step 6** (chỉ khác engine):
```
input  : Keyframes/<batch>/<video_name>/<idx>.webp
output : OCR/<video_name>.json   # { "<frame_id>": "<ocr text>", ... }
         ocr_data.json           # merged flat dict, drop-in cho backend/data/ocr/
```

**Cost**: ~0.8–1.2 s/frame với 3B + fp16 trên T4; 2 GPU song song ≈ ~0.5 s/frame hiệu
dụng. Toàn bộ ~55k frames ≈ 2 phiên 12h. Bật `FRAME_STRIDE = 2` để phủ 1/2 chi phí.

**Resumable 2 cấp**:
- *per video* - `OCR/<video_name>.json` đủ frame thì bỏ qua (kể cả khi chạy lại)
- *per frame* - trong video, frame đã có thì không re-read, checkpoint mỗi `CHECKPOINT_EVERY`

**Accelerator**: GPU T4 x2. (Nếu chỉ có 1 GPU, đổi `NUM_GPUS = 1`.)


In [ ]:
# Qwen2.5-VL cần transformers mới + qwen-vl-utils (helper xử lý ảnh)
!pip install -q -U "transformers>=4.49" qwen-vl-utils accelerate


In [ ]:
import gc
import json
import os
import re
import subprocess
import sys
import time
from pathlib import Path

import torch
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

print('torch', torch.__version__,
      '| GPUs:', torch.cuda.device_count() if torch.cuda.is_available() else 0)


In [ ]:
# === INPUTS - sửa nếu keyframe dataset đặt tên khác =====================
KEYFRAMES_DIR    = '/kaggle/input/aic-keyframes-webp/Keyframes'
OUTPUT_DIR       = './OCR'                 # one <video_name>.json per video (same as step 6)

# === SONG SONG 2 GPU ====================================================
NUM_GPUS         = 2        # số T4 được cấp; đổi thành 1 nếu chỉ có 1 GPU
CONFIG_FILE      = '_ocr_config.json'      # config dump cho worker
WORKER_SCRIPT    = '_ocr_worker.py'        # script worker, sinh tự động ở cell sau

# === MODEL - đổi sang 7B nếu có GPU >= 24GB (A10/A100) ==================
#   3B : mặc định, vừa T4 16GB fp16 (~7GB VRAM), nhanh, OCR tốt
#   7B : chính xác hơn, đặc biệt với text nhỏ / banner phức tạp
MODEL_ID         = 'Qwen/Qwen2.5-VL-3B-Instruct'
# MODEL_ID       = 'Qwen/Qwen2.5-VL-7B-Instruct'

# === TUNING ============================================================
FRAME_STRIDE     = 1        # 1 = mọi keyframe; 2 = mỗi 2 frame (giảm 1/2 cost)
MAX_NEW_TOKENS   = 160      # OCR tiếng Việt banner thường < 80 token, để dư 1 chút
IMG_BATCH        = 4        # ảnh / 1 lần generate() - gộp batch tăng throughput (test 2/4/8)
IMAGE_EXTS       = ('.jpg', '.jpeg', '.png', '.webp')
CHECKPOINT_EVERY = 32       # flush JSON mỗi N frame trong 1 video
MAX_IMAGE_SIDE   = 768      # downscale ảnh to hơn cạnh này -> nhanh hơn, OCR vẫn OK

# === PROMPT - gọn, ép JSON để parse ổn định ============================
PROMPT = (
    "You are reading the text visible in one keyframe from a Vietnamese TV "
    "broadcast.\n"
    "Transcribe ALL on-screen text verbatim. Keep Vietnamese diacritics exactly "
    "as shown. If multiple text elements exist (banner, logo, lower-third, ticker, "
    "caption), concatenate them with a single space, in reading order "
    "(top to bottom, left to right). If there is NO visible text, return an "
    "empty string.\n"
    "Return ONLY a JSON object: {\"ocr\": \"<the text>\"}. No markdown, no "
    "explanation, no code fence."
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


In [ ]:
# Mỗi worker load model 1 lần trên GPU riêng → dump config + sinh worker script.
_config = {
    'keyframes_dir': str(Path(KEYFRAMES_DIR)),
    'output_dir': str(Path(OUTPUT_DIR)),
    'model_id': MODEL_ID,
    'frame_stride': FRAME_STRIDE,
    'max_new_tokens': MAX_NEW_TOKENS,
    'image_batch': IMG_BATCH,
    'image_exts': list(IMAGE_EXTS),
    'checkpoint_every': CHECKPOINT_EVERY,
    'max_image_side': MAX_IMAGE_SIDE,
    'prompt': PROMPT,
}
with open(CONFIG_FILE, 'w', encoding='utf-8') as f:
    json.dump(_config, f, ensure_ascii=False, indent=2)
print('config ->', CONFIG_FILE)

WORKER_SRC = r'''import argparse
import gc
import json
import os
import re
import sys
import time
from pathlib import Path

from tqdm import tqdm

import torch
from PIL import Image
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

CONFIG_PATH = '_ocr_config.json'
with open(CONFIG_PATH, encoding='utf-8') as f:
    cfg = json.load(f)

KEYFRAMES_DIR    = cfg['keyframes_dir']
OUTPUT_DIR       = cfg['output_dir']
MODEL_ID         = cfg['model_id']
FRAME_STRIDE     = cfg['frame_stride']
MAX_NEW_TOKENS   = cfg['max_new_tokens']
IMG_BATCH        = cfg['image_batch']
IMAGE_EXTS       = tuple(cfg['image_exts'])
CHECKPOINT_EVERY = cfg['checkpoint_every']
MAX_IMAGE_SIDE   = cfg['max_image_side']
PROMPT           = cfg['prompt']

BATCH_PROMPT = (
    "You are reading the text visible in several keyframes from a Vietnamese TV "
    "broadcast. The images are given in order.\n"
    "For EACH image, transcribe ALL on-screen text verbatim. Keep Vietnamese "
    "diacritics exactly as shown. If multiple text elements exist (banner, logo, "
    "lower-third, ticker, caption), concatenate them with a single space, in "
    "reading order (top to bottom, left to right). If an image has NO visible "
    "text, use an empty string.\n"
    "Return ONLY a JSON array with exactly the same number of objects as images, "
    "in the same order: [{\"ocr\": \"<text image 1>\"}, {\"ocr\": \"<text image 2>\"}, ...]. "
    "No markdown, no explanation, no code fence."
)

os.makedirs(OUTPUT_DIR, exist_ok=True)


def _parse_ocr_json(text):
    m = re.search(r'\{.*?\}', text, flags=re.DOTALL)
    if not m:
        return ''
    try:
        d = json.loads(m.group(0))
    except ValueError:
        return ''
    val = d.get('ocr', '')
    return str(val).strip() if val else ''


def _parse_ocr_list(text, n):
    m = re.search(r'\[.*?\]', text, flags=re.DOTALL)
    if not m:
        return None
    try:
        d = json.loads(m.group(0))
    except ValueError:
        return None
    if not isinstance(d, list) or len(d) != n:
        return None
    out = []
    for item in d:
        if isinstance(item, dict):
            out.append(str(item.get('ocr', '')).strip())
        else:
            out.append(str(item).strip())
    return out


def _load_image(path):
    img = Image.open(path).convert('RGB')
    w, h = img.size
    m = max(w, h)
    if m > MAX_IMAGE_SIDE:
        scale = MAX_IMAGE_SIDE / m
        img = img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)
    return img


@torch.inference_mode()
def ocr_image(path, model, processor):
    image = _load_image(path)
    messages = [{'role': 'user', 'content': [
        {'type': 'image', 'image': image},
        {'type': 'text', 'text': PROMPT},
    ]}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[text], images=[image], return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
    )
    reply = processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]
    return _parse_ocr_json(reply)


@torch.inference_mode()
def ocr_images(paths, model, processor):
    images = [_load_image(p) for p in paths]
    messages = [{'role': 'user', 'content': (
        [{'type': 'image', 'image': im} for im in images] +
        [{'type': 'text', 'text': BATCH_PROMPT}]
    )}]
    text = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    inputs = processor(text=[text], images=images, return_tensors='pt').to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS * len(images),
        do_sample=False,
    )
    reply = processor.batch_decode(
        out[:, inputs.input_ids.shape[1]:], skip_special_tokens=True
    )[0]
    return _parse_ocr_list(reply, len(images))


def out_path(video_name):
    return Path(OUTPUT_DIR) / f'{video_name}.json'


def load_video_ocr(video_name):
    p = out_path(video_name)
    if p.exists():
        with p.open(encoding='utf-8') as f:
            return json.load(f)
    return {}


def save_video_ocr(video_name, data):
    with out_path(video_name).open('w', encoding='utf-8') as f:
        json.dump(data, f, ensure_ascii=False)


def build_video_index():
    root = Path(KEYFRAMES_DIR)
    videos = {}
    for img in root.rglob('*'):
        if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
            video_name = img.parent.name
            frame_id = f'{video_name}_{img.stem}'
            videos.setdefault(video_name, []).append((frame_id, img))
    for v in videos:
        videos[v].sort(key=lambda t: t[0])
    return videos


def load_model(device):
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.float16 if device.startswith('cuda') else torch.float32,
        device_map=device,
        attn_implementation='sdpa',
    )
    processor = AutoProcessor.from_pretrained(MODEL_ID)
    model.eval()
    return model, processor


def run_smoke(device):
    first = next(
        p for p in sorted(Path(KEYFRAMES_DIR).rglob('*'))
        if p.suffix.lower() in IMAGE_EXTS
    )
    print('smoke frame:', first, flush=True)
    model, processor = load_model(device)
    t0 = time.time()
    text = ocr_image(first, model, processor)
    print(f'inference: {time.time() - t0:.2f}s', flush=True)
    print('OCR ->', repr(text[:200]), flush=True)


def run_corpus(device, video_names):
    model, processor = load_model(device)
    print(f'[{device}] model ready: {MODEL_ID}', flush=True)
    videos = build_video_index()
    total_done = 0
    t_start = time.time()
    for video_name in video_names:
        frames = videos[video_name][::FRAME_STRIDE]
        data = load_video_ocr(video_name)
        todo = [(fid, p) for fid, p in frames if fid not in data]
        if not todo:
            total_done += len(frames)
            continue
        pbar = tqdm(total=len(todo), desc=f'[{device}] {video_name}', unit='img',
                    mininterval=1.0, file=sys.stdout, leave=False)
        done_in_video = 0
        for i in range(0, len(todo), IMG_BATCH):
            chunk = todo[i:i + IMG_BATCH]
            paths = [p for _, p in chunk]
            try:
                texts = ocr_images(paths, model, processor)
                if texts is None:                      # model trả sai format -> fallback từng ảnh
                    texts = [ocr_image(p, model, processor) for _, p in chunk]
            except Exception as e:
                print(f'[WARN] batch failed: {type(e).__name__}: {e}', flush=True)
                texts = [ocr_image(p, model, processor) for _, p in chunk]
            for (fid, _p), t in zip(chunk, texts):
                data[fid] = t
            done_in_video += len(chunk)
            pbar.update(len(chunk))
            if done_in_video % CHECKPOINT_EVERY == 0:
                save_video_ocr(video_name, data)
        pbar.close()
        save_video_ocr(video_name, data)
        total_done += len(frames)
        _el = time.time() - t_start
        print(
            f'[{device}] {video_name}: {len(frames)} frames | '
            f'cum {total_done} | {_el:.1f}s | {total_done / max(_el, 1e-6):.2f} img/s',
            flush=True,
        )
    _el = time.time() - t_start
    print(
        f'[{device}] done {total_done} frames in {_el / 60:.1f} min '
        f'| {total_done / max(_el, 1e-6):.2f} img/s',
        flush=True,
    )


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--gpu', type=int, default=-1)
    ap.add_argument('--videos', nargs='*')
    ap.add_argument('--smoke', action='store_true')
    args = ap.parse_args()
    device = f'cuda:{args.gpu}' if args.gpu >= 0 else 'cpu'
    if args.smoke:
        run_smoke(device)
    else:
        run_corpus(device, sorted(args.videos))


if __name__ == '__main__':
    main()
'''
with open(WORKER_SCRIPT, 'w', encoding='utf-8') as f:
    f.write(WORKER_SRC)
print('worker script ->', WORKER_SCRIPT)


In [ ]:
# Smoke test qua worker trên GPU 0 - sanity check trước khi chạy cả corpus.
cmd = [sys.executable, WORKER_SCRIPT, '--gpu', '0', '--smoke']
r = subprocess.run(cmd, capture_output=True, text=True,
                   encoding='utf-8', errors='replace')
print(r.stdout)
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise SystemExit('smoke test failed')


In [ ]:
# Group mọi keyframe theo video: {video_name: [(frame_id, path), ...]}
root = Path(KEYFRAMES_DIR)
videos = {}
for img in root.rglob('*'):
    if img.is_file() and img.suffix.lower() in IMAGE_EXTS:
        video_name = img.parent.name                  # e.g. L21_V001
        frame_id   = f'{video_name}_{img.stem}'       # img.stem = zero-padded idx
        videos.setdefault(video_name, []).append((frame_id, img))
for v in videos:
    videos[v].sort(key=lambda t: t[0])

print('videos:', len(videos),
      '| frames:', sum(len(v) for v in videos.values()),
      '| stride:', FRAME_STRIDE)


In [ ]:
# Chia video round-robin cho NUM_GPUS worker, chạy 2 tiến trình song song.
video_names = sorted(videos)
chunks = [video_names[i::NUM_GPUS] for i in range(NUM_GPUS)]
for g in range(NUM_GPUS):
    print(f'gpu{g}: {len(chunks[g])} videos')

t_start = time.time()
procs = []
for gpu, vlist in enumerate(chunks):
    if not vlist:
        continue
    cmd = [sys.executable, WORKER_SCRIPT, '--gpu', str(gpu), '--videos', *vlist]
    p = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        encoding='utf-8',
        errors='replace',
    )
    procs.append(p)

# Stream output 2 worker live rồi đợi cả 2 xong.
# Đọc theo ký tự để hiểu được \r của tqdm (progress bar hiện ngay trên notebook).
def _stream(proc):
    buf = ''
    while True:
        ch = proc.stdout.read(1)
        if ch == '':
            break
        if ch == '\n':
            print(buf, flush=True)
            buf = ''
        elif ch == '\r':
            print('\r' + buf, end='', flush=True)
            buf = ''
        else:
            buf += ch
    if buf:
        print(buf, flush=True)

for p in procs:
    _stream(p)
    rc = p.wait()
    if rc != 0:
        raise SystemExit(f'worker exited with code {rc}')
elapsed = time.time() - t_start
print(f'\nall workers done in {elapsed/60:.1f} min')


In [ ]:
# Merge các file per-video thành 1 file flat đúng format backend.
# (Khi copy về local: file này sẽ thay thẳng backend/data/ocr/ocr_data.json.)
merged = {}
for jf in sorted(Path(OUTPUT_DIR).glob('*.json')):
    with jf.open(encoding='utf-8') as f:
        merged.update(json.load(f))

with open('ocr_data.json', 'w', encoding='utf-8') as f:
    json.dump(merged, f, ensure_ascii=False)

print(f'merged entries: {len(merged)} -> ocr_data.json')
print(f'non-empty:      {sum(1 for v in merged.values() if v)} '
      f'({sum(1 for v in merged.values() if v)/max(len(merged),1)*100:.1f}%)')


## Dùng với backend

Sau khi notebook chạy xong, download 2 file về và copy đúng chỗ:

```bash
# 1) OCR mới thay thẳng file EasyOCR cũ (drop-in, format giống hệt)
cp ocr_data.json  backend/data/ocr/ocr_data.json

# 2) Rebuild lại các index phụ thuộc OCR (scene text trong graph cũng đổi):
cd data_processing
python 07_build_text_indexes.py     # ocr_text + asr_text + caption_text index
python 08_build_graph.py            # scene.ocr lấy từ OCR mới
```

Backend tự nhận - không cần sửa code:
- `/search_ocr` dùng OCR sạch hơn (Qwen2.5-VL)
- `/search_text` với `scene_keywords` ăn theo scene.ocr mới
- Object detection / graph vẫn dùng detections cũ, không bị ảnh hưởng
